In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: local


In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q transformers==5.5.4
    !pip install -q minigrid==3.0.0
    !pip install -q langchain_huggingface==1.2.2

In [2]:
import os
import sys

if EXECUTION_ENV == "colab":
    # 1. Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Setup path
    PROJECT_DIR = '/content/drive/My Drive/EAD-Pesquisa-Agentes-codigo/_projeto_minigrid'
    if not os.path.exists(PROJECT_DIR):
        os.makedirs(PROJECT_DIR)
    sys.path.append(PROJECT_DIR)

if EXECUTION_ENV == "kaggle":
    sys.path.append('/kaggle/input/datasets/pabloazevedo/base-src')

In [3]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

from react_agent import ReActAgent
from wrappers import SYSTEM_PROMPT_WRAPPER_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE, OBS_TEMPLATE_ENG

from experiments_util import run_and_save_experiments, wrapper1, wrapper2_with_numbers, wrapper2_without_numbers

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
if EXECUTION_ENV == "colab":
    from google.colab import userdata
    if userdata.get("HF_TOKEN"):
        os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")
    else:
        raise Exception("A secret named HF_TOKEN must be set in Colab")

if EXECUTION_ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")

if EXECUTION_ENV == "local":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN is set
    assert os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_API_KEY"), "Either HUGGINGFACE_API_KEY or HF_TOKEN environment variable must be set"


## Model Configuration

In [4]:
# Ele efetivamente usa 2 bilhões de parâmetros por vez (note o "2B" no nome), apesar de possuir mais.
# Model card: https://huggingface.co/google/gemma-4-E2B-it
#MODEL_ID = "google/gemma-4-E2B-it"

# Modelos a serem testados
MODEL_ID = "google/gemma-3-1b-it"
#MODEL_ID = "google/gemma-3-4b-it"
#MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
#MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # non-thinking version
#MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
#MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"  # english only!
#MODEL_ID = "microsoft/Phi-3.5-mini-instruct"   # multi-lingual
#MODEL_ID = "HuggingFaceTB/SmolLM3-3B"

In [ ]:
local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs=dict(
        do_sample=True,
        max_new_tokens=2048,
        return_full_text=False  # Atenção: Precisa setar este valor para contornar um bug!!!
    ),
)

In [ ]:
HF_CHAT_MODEL = ChatHuggingFace(llm=local_llm,max_retries=2)

In [7]:
HF_CHAT_MODEL = None

## Experiment Configuration

In [5]:
model_name = MODEL_ID.split("/")[-1]
print(f"Model name: {model_name}")

Model name: gemma-3-1b-it


In [8]:
# PARTE 1
configs = [
    {
        'name': f'Prompt_2d_{model_name}_without_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': f'Prompt_2d_{model_name}_with_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
]


In [ ]:
# PARTE 2
'''
configs.extend([
    {
        'name': f'Prompt_1_{model_name}_without_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': f'Prompt_1_{model_name}_with_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
])
#''';

## Running Experiment

In [ ]:
# Use um nome novo ou None para iniciar um novo experimento.
# Ou repita um nome já existente na pasta "results" para continuar um experimento.
experiment_name = "Lorraine"

In [ ]:
##########################
##  Execute experiments ##
##########################

if EXECUTION_ENV == "kaggle":
    import experiments_util
    experiments_util.RESULTS_BASE_DIR = "/kaggle/working/results"

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)


In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory 
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything 
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

## Plot Results

In [ ]:
print("Now, run notebook \"run_experiments_analysis.ipynb\".")